# `002 / ExaptGNN.v1`

In [ ]:
COMMIT = ""
DATA_REV = ""
MANIFEST = ""

STAGE = "pretrain"
PREFLIGHT = ""
LAUNCH = False
RESUME = False
STOP_AFTER = None

PRETRAIN = [
    "--steps", "2000", "--width", "128",
    "--batch-size", "8192", "--eval-every", "50",
    "--patience", "8", "--weight-decay", "0.0001",
    "--learning-rate", "0.001",
    "--loss-min-delta", "0.0001",
    "--checkpoint-every", "100",
]
FIT = [
    "--epochs", "500", "--width", "128",
    "--learning-rate", "0.0005", "--negatives", "8",
    "--eval-every", "5", "--prototype-k", "3",
    "--patience", "15", "--weight-decay", "0.0001",
    "--min-epochs", "50", "--map-min-delta", "0.0001",
]

In [ ]:
import hashlib
import os
import re
import subprocess
import sys
from pathlib import Path

from google.colab import userdata


def run(*args, cwd=None):
    subprocess.run([str(arg) for arg in args], cwd=cwd, check=True)


for description, value, length in (
    ("source commit", COMMIT, 40),
    ("dataset revision", DATA_REV, 40),
    ("manifest checksum", MANIFEST, 64),
):
    if re.fullmatch(rf"[0-9a-f]{{{length}}}", value) is None:
        raise ValueError(f"{description} must have {length} lowercase hex digits")

token = userdata.get("exapt_hf")
if not token:
    raise RuntimeError("hugging face token missing from colab secrets")
os.environ["HF_TOKEN"] = token
del token

repo = Path("/content/exapt")
if not repo.exists():
    run("git", "clone", "https://github.com/aekn/exapt.git", repo)
if subprocess.check_output(
    ["git", "status", "--porcelain"], cwd=repo, text=True
).strip():
    raise RuntimeError("source checkout has uncommitted changes")
run("git", "checkout", "--detach", COMMIT, cwd=repo)
if subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=repo, text=True
).strip() != COMMIT:
    raise RuntimeError("source checkout does not match the pinned commit")

run(sys.executable, "-m", "pip", "install", "-q", "uv")
run("uv", "sync", "--locked", "--python", "3.12", cwd=repo)
run("uv", "run", "python", "-c", "import torch; "
    "assert torch.cuda.is_available(), 'CUDA unavailable'; "
    "print(torch.__version__, torch.version.cuda, "
    "torch.cuda.get_device_name(0))", cwd=repo)

prepared = repo / "data/processed"
run("uv", "run", "exapt", "hub", "download",
    "exapt-lab/data", DATA_REV, prepared, cwd=repo)
actual = hashlib.sha256((prepared / "manifest.json").read_bytes()).hexdigest()
if actual != MANIFEST:
    raise RuntimeError("dataset manifest checksum does not match")
run("uv", "run", "python", "-m", "unittest", "discover",
    "-s", "tests", "-q", cwd=repo)

In [ ]:
if STAGE not in ("pretrain", "fit"):
    raise ValueError(
        f"unknown training stage {STAGE!r}; use pretrain or fit"
    )
if PREFLIGHT not in ("", "overfit", "profile"):
    raise ValueError(
        f"unknown preflight mode {PREFLIGHT!r}; use overfit or profile"
    )
if PREFLIGHT == "overfit":
    run("uv", "run", "exapt", "preflight", "overfit",
        prepared / "split/smoke", "--width", "32", "--steps", "300",
        "--prototype-k", "3", "--device", "cuda", cwd=repo)
elif PREFLIGHT == "profile":
    if STAGE == "pretrain":
        command = ["profile-pretrain", prepared / "pretrain.pt",
                   repo / "runs/preflight-002-pretrain"]
    else:
        command = ["profile-fit", prepared / "graph.pt",
                   prepared / "split", repo / "runs/preflight-002-fit",
                   "--prototype-k", "3"]
    run("uv", "run", "exapt", "preflight", *command,
        "--width", "128", "--device", "cuda", cwd=repo)
elif not LAUNCH:
    print(f"Ready: 002 / {STAGE}. Training disabled.")
else:
    output = repo / "runs/002" / STAGE
    flags = []
    if RESUME:
        if not (output / "latest.json").is_file():
            run("uv", "run", "exapt", "hub", "recover",
                "exapt-lab/hangar", "002", STAGE, output, cwd=repo)
        flags += ["--resume", str(output / "latest.json")]
    if STOP_AFTER is not None:
        flags += ["--stop-after", str(STOP_AFTER)]

    if STAGE == "pretrain":
        command = ["pretrain", prepared / "pretrain.pt", output, *PRETRAIN]
    else:
        encoder = repo / "runs/002/pretrained.pt"
        run("uv", "run", "exapt", "hub", "selected",
            "exapt-lab/hangar", "002", "pretrain", encoder, cwd=repo)
        command = ["train", prepared / "graph.pt", prepared / "split",
                   output, "--pretrained", encoder, *FIT]

    print(f"Run: 002 / {STAGE}")
    run("uv", "run", "exapt", *command, "--device", "cuda",
        "--run-id", "002", "--bucket", "exapt-lab/hangar",
        "--source-commit", COMMIT, "--dataset-revision", DATA_REV,
        "--manifest-sha256", MANIFEST, *flags, cwd=repo)